# W12: FnB Cockpit

도입 전/후 KPI를 비교해 막대그래프와 경영진 보고 문안을 자동 생성합니다.


In [ ]:
import io
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm

try:
    import google.generativeai as genai
except Exception:
    genai = None

GEMINI_MODEL = "gemini-2.0-flash"

for fp in fm.findSystemFonts():
    try:
        fm.fontManager.addfont(fp)
    except Exception:
        continue


def safe_upload():
    try:
        from google.colab import files
    except Exception:
        print("Colab 환경이 아니라 files.upload를 사용할 수 없습니다.")
        return {}
    try:
        return files.upload()
    except Exception as e:
        print(f"files.upload() 취소/실패: {e}")
        print("files.upload() 취소되어 기본 데이터 사용")
        return {}


def use_gemini(prompt: str, fallback: str) -> str:
    if not genai:
        return fallback
    key = os.getenv("GOOGLE_API_KEY", "").strip()
    if not key:
        return fallback
    try:
        genai.configure(api_key=key)
        model = genai.GenerativeModel(GEMINI_MODEL)
        result = model.generate_content(prompt)
        return getattr(result, "text", "").strip() or fallback
    except Exception as e:
        print(f"Gemini 호출 실패: {e}")
        return fallback

uploaded = safe_upload()
if uploaded:
    fn = list(uploaded.keys())[0]
    kpi = pd.read_csv(io.StringIO(uploaded[fn].decode("utf-8"))
)
else:
    kpi = pd.DataFrame(
        {
            "지표": ["주문전환율", "평균객단가", "재구매율", "클레임율"],
            "도입전": [0.11, 12000, 0.28, 0.05],
            "도입후": [0.16, 13800, 0.36, 0.03],
        }
    )

idx = np.arange(len(kpi))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(idx - w/2, kpi["도입전"], w, label="도입전")
ax.bar(idx + w/2, kpi["도입후"], w, label="도입후")
ax.set_xticks(idx)
ax.set_xticklabels(kpi["지표"], rotation=10)
ax.set_title("도입 전/후 KPI 비교")
ax.legend()
plt.tight_layout()
plt.show()

kpi["개선율"] = ((kpi["도입후"] - kpi["도입전"]) / kpi["도입전"] * 100).round(2)
print(kpi)
rows = [f"{r.지표}: {r.도입전} -> {r.도입후} ({r.개선율}%)" for r in kpi.itertuples()]
report = use_gemini(
    "경영진 보고서 톤으로, 아래 성과를 요약하고 다음 분기 액션을 2개 제시해줘:
" + "
".join(rows),
    "전체적으로 KPI가 개선되었고, 특히 주문전환율 및 재구매율 상승이 돋보입니다. 하위클레임 관리와 프로모션 효율 점검을 우선 시행합니다."
)
print(report)
